# Case Study – Data Modeling in Redis

## Setup

### Running Redis in Docker

- [Docker Hub: Redis Stack Server](https://hub.docker.com/r/redis/redis-stack-server)

In [ ]:
# !docker pull redis/redis-stack-server
# !docker run -d --name redis -p 6379:6379 -p 8001:8001 redis/redis-stack-server:latest

### Imports

In [ ]:
from redis import Redis
from enum import Enum
from typing import Optional
from dataclasses import dataclass
from datetime import datetime

### Data Structures

In [ ]:
@dataclass
class User:
    id: str
    name: str

@dataclass
class Item:
    id: str
    name: str
    price: float

### Enums

In [ ]:
class ActionType(Enum):
    ADD_TO_CART = "added_to_cart"
    REMOVE_FROM_CART = "removed_from_cart"
    VIEW_ITEM = "viewed_item"
    CHECKOUT = "checkout"

### Class: `Ecommerce`

In [ ]:
class Ecommerce:
    def __init__(self, host: str, port: int) -> None:
        """
        Initializes the `Ecommerce` class.

        :param host: The Redis database host (eg. "localhost").
        :type host: str

        :param port: The Redis database port (eg. 6379).
        :type port: int
        """

        self.r: Redis = Redis(host=host, port=port, db=0, decode_responses=True)


    def session(self, user: User) -> None:
        """
        Verifies if a given user is logged in.\n
        If not, starts a new session with a TTL of 30 minutes.

        :param user: The user reference to verify the session.
        :type user: User
        """

        session_key = f"session:{user.name}"
        any_user = self.r.hgetall(name=session_key)

        if not any_user:
            self.r.hset(name=session_key, mapping={ "id": user.id, "name": user.name, "login_time": str(datetime.now()) })
            self.r.expire(name=session_key, time=1800)  # 30min
            print(f"User '{user.name}' logged in!")
        else:
            current_ttl = self.r.ttl(name=session_key)
            print(f"User '{user.name}' already logged in. Remaining time in session: {current_ttl} seconds.")


    def _update_user_activity(self, user: User, action_type: ActionType, item: Optional[Item]=None) -> None:
        """
        Records each action performed by the given user.

        :param user: The respective user to have its actions recorded.
        :type user: User

        :param action_type: The type of the action performed.
        :type action_type: ActionType

        :param item: The item that complements the respective action, if any.
        :type item: Optional[Item]
        """

        history_key = f"history:{user.id}"
        action = ':'.join([action_type.value, item.id if item else ""])
        self.r.lpush(history_key, action)
        self.r.ltrim(name=history_key, start=0, end=49)  # Keep only the last 50 records
        self.r.hset(name=f"session:{user.name}", key="last_activity", value=str(datetime.now()))


    def handle_shopping_cart(self, user: User, item: Item, amount: int) -> None:
        """
        Adds or removes an item to the given user's shopping cart.\n
        If the item's amount reaches 0 or bellow, it is completely removed from the cart.\n
        This operation can only be performed if the given user has an active session.

        :param user: The user reference to handle the shopping cart.
        :type user: User

        :param item: The item to be handled in the user's shopping cart.
        :type item: Item

        :param amount: The amount of the respective item. Negative values indicate removal.
        :type amount: int
        """

        session_key = f"session:{user.name}"
        any_user = self.r.hgetall(name=session_key)

        if any_user:
            user_id = self.r.hget(name=session_key, key="id")
            shopping_cart_key = f"cart:{user_id}"
            item_key = f"item:{item.id}"
            current_amount = self.r.hincrby(name=shopping_cart_key, key=item_key, amount=amount)

            if amount > 0:
                print(f"Adding {amount} '{item.name}' to the cart.")
                self._update_user_activity(user=user, action_type=ActionType.ADD_TO_CART, item=item)
            elif current_amount <= 0:  # type: ignore (Pylance may complain here)
                self.r.hdel(shopping_cart_key, item_key)
                print(f"Item '{item.name}' completely removed from the cart.")
                self._update_user_activity(user=user, action_type=ActionType.REMOVE_FROM_CART, item=item)
            else:
                print(f"Removing {amount} '{item.name}' from the cart.")
                self._update_user_activity(user=user, action_type=ActionType.REMOVE_FROM_CART, item=item)
        else:
            print(f"User '{user.name}' has not an active session.")


    def view_item(self, user: User, item: Item) -> None:
        """
        Shows the details of an item.

        :param user: The user reference that is viewing the item.
        :type user: User

        :param item: The respective item to show its details.
        :type item: Item
        """

        session_key = f"session:{user.name}"
        any_user = self.r.hgetall(name=session_key)

        if any_user:
            print(f"ID: {item.id} | Name: {item.name} | Price: {item.price}")
            self._update_user_activity(user=user, action_type=ActionType.VIEW_ITEM, item=item)
        else:
            print(f"User '{user.name}' has not an active session.")


    def view_user_history(self, user: User) -> None:
        """
        Shows the entire history of a given user.

        :param user: The user reference to show the history.
        :type user: User
        """

        history = self.r.lrange(name=f"history:{user.id}", start=0, end=-1)  # Shows the entire history
        [print(record) for record in history]  # type: ignore (Pylance may complain here)


    def checkout(self, user: User) -> None:
        """
        Performs a checkout in the E-comerce, of which consists of:
        - Register a sale for each item in the given user's shopping cart
        - Empty the given user's shopping cart

        :param user: The user reference to perform the checkout action.
        :type user: User
        """

        shopping_cart_key = f"cart:{user.id}"
        shopping_cart_items = self.r.hgetall(name=shopping_cart_key)

        for key, value in shopping_cart_items.items():  # type: ignore (Pylance may complain here)
            self.r.zadd(name="item:sales", mapping={ str(key).split(':')[1]: value })

        # self.r.delete(shopping_cart_key)
        self._update_user_activity(user=user, action_type=ActionType.CHECKOUT)
        print("Checkout successful!")


    def view_sales(self) -> None:
        """
        Displays the top 10 best-selling items from the entire E-comerce.
        """

        print(self.r.zrange(name="item:sales", start=0, end=9, withscores=True))

### Mock Data

#### Users

In [ ]:
user1 = User(
    id="u_01",
    name="john_doe"
)

#### Items

In [ ]:
item1 = Item(
    id="i_01",
    name="Keyboard",
    price=149.9
)

item2 = Item(
    id="i_02",
    name="Mouse",
    price=99.9
)

item3 = Item(
    id="i_03",
    name="USB-C to USB-A Adapter",
    price=19.9
)

item4 = Item(
    id="i_04",
    name="Monitor",
    price=299.9
)

## Actions

Create E-commerce

In [ ]:
ecommerce = Ecommerce(
    host="localhost",
    port=6379
)

Start new session

In [ ]:
ecommerce.session(user=user1)

View items

In [ ]:
ecommerce.view_item(user=user1, item=item1)
ecommerce.view_item(user=user1, item=item2)
ecommerce.view_item(user=user1, item=item3)

View history

In [ ]:
ecommerce.view_user_history(user=user1)

Add items to shopping cart

In [ ]:
ecommerce.handle_shopping_cart(user=user1, item=item1, amount=1)
ecommerce.handle_shopping_cart(user=user1, item=item2, amount=1)
ecommerce.handle_shopping_cart(user=user1, item=item3, amount=3)
ecommerce.handle_shopping_cart(user=user1, item=item4, amount=1)

View history after adding items to cart

In [ ]:
ecommerce.view_user_history(user=user1)

Remove items from shopping cart

In [ ]:
ecommerce.handle_shopping_cart(user=user1, item=item1, amount=-1)
ecommerce.handle_shopping_cart(user=user1, item=item3, amount=-1)

Perform checkout

In [ ]:
ecommerce.checkout(user=user1)

View history after checkout

In [ ]:
ecommerce.view_user_history(user=user1)

View current sales

In [ ]:
ecommerce.view_sales()


---

[pt-BR]

## Responder

Q1: **Justificar Redis para cache e sessão.**

R: Quando o assunto é cache e gerenciamento de sessões, o Redis é, praticamente, a escolha padrão no mercado. Isso se dá pelo fato desse banco de dados combinar **velocidade absurda** com **alta facilidade de utilização**. Diferente dos bancos de dados tradicionais, que utilizam de armazenamento em disco, o Redis armazena tudo em memória RAM. Com isso, é possível garantir uma latência extretamente baixa (questão de microsegundos), além de possibilitar uma taxa muito alta de operações por segundos (cerca de centenas de milhares ou milhões).

Q2: **Modelar um carrinho com 3 produtos usando comandos Redis.**

(*demonstrado anteriormente*)

Q3: **Explicar a diferença entre *List* e *Stream*.**

R: Uma lista no Redis é basicamente uma **lista duplamente encadeada**, onde seus elementos são ordenados por *inserção*. Algumas das principais características de uma lista são:
- Ótimas para filas simples (*queues*) ou pilhas (*stacks*)
- São muito rápidas
- Possuem "consumo destrutivo" (perde o item ao removê-lo, a exemplo de utilizar o comando `LPOP`)

Cenários indicados para utilização de listas:
- Necessidade de uma fila simples
- Ordem FIFO (fist in, first out) e LIFO (last in, first out) é suficiente
- Não precisa de histórico

Já o stream se trata de uma estrutura mais moderna, sendo bastante recomendado em casos como:
- Processamento de eventos
- Múltiplos consumidores
- Reprocessamento seguro

Cenários indicados para utilização de streams:
- Log de eventos
- Reprocessamento de mensagens em casos de falha
- Sistemas com arquitetura próxima de um sistema de mensageria

Q4: **Criar uma solução para armazenar e consultar os produtos mais visualizados na última hora.**

(*demonstrado anteriormente*)